# Kaggle Precompute Pipeline: WAV -> Mel `.npy` + Dataset Publish

This notebook:
1. Reads all audio files referenced by `train_split.csv`, `val_split.csv`, and `test_split.csv`.
2. Converts each file to mel spectrogram and saves to `.npy`.
3. Writes updated split CSVs that point to mel `.npy` paths.
4. Packages outputs and publishes a new Kaggle dataset (create or version).

Run this on Kaggle with Internet enabled if you want automatic dataset upload.

## 1) Install and imports

In [ ]:
import json
import os
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torchaudio
import torchaudio.transforms as T

def ensure_kaggle_cli():
    try:
        subprocess.check_call(["kaggle", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    except Exception:
        subprocess.check_call(["python", "-m", "pip", "install", "-q", "kaggle"])

ensure_kaggle_cli()
print("Imports ready.")

## 2) Configuration

In [ ]:
@dataclass
class CFG:
    # Input dataset root (Kaggle mount or local path)
    dataset_root: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset"
    data_root: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/data"

    # Split control: if True and CSVs exist, use them; otherwise derive splits from folders.
    use_csv_splits_if_available: bool = True
    train_csv: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/train_split.csv"
    val_csv: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/val_split.csv"
    test_csv: str = "/kaggle/input/datasets/karimmyasser/machine-large-dataset/test_split.csv"
    split_seed: int = 42
    train_ratio: float = 0.7
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    # Mel settings
    target_sr: int = 22050
    clip_seconds: float = 5.0
    n_fft: int = 1024
    hop_length: int = 512
    n_mels: int = 128

    # Preprocessing controls
    remove_dc_offset: bool = True
    peak_normalize_waveform: bool = True
    pre_emphasis: bool = False
    pre_emphasis_coef: float = 0.97
    mel_floor_db: float = -80.0
    mel_ceil_db: float = 0.0
    mel_scale_mode: str = "minmax"  # minmax or zscore

    # Storage
    out_root: str = "/kaggle/working/mel_npy_dataset"
    save_dtype: str = "float16"  # float16 or float32

    # Kaggle publish config
    publish: bool = True
    kaggle_owner: str = "karimmyasser"
    kaggle_dataset_slug: str = "machine-sound-mel-npy"
    kaggle_title: str = "Machine Sound Mel NPY"
    kaggle_subtitle: str = "Precomputed mel spectrograms and split CSVs"
    kaggle_description: str = "Precomputed mel spectrogram .npy files generated from WAV files with train/val/test split manifests."
    kaggle_license: str = "CC0-1.0"

cfg = CFG()
Path(cfg.out_root).mkdir(parents=True, exist_ok=True)
print(json.dumps(asdict(cfg), indent=2))

## 3) Helpers

In [ ]:
dataset_root = Path(cfg.dataset_root)
data_root = Path(cfg.data_root)

for p in [dataset_root, data_root]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required path: {p}")

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    path_candidates = ["file_path", "filepath", "path", "audio_path"]
    label_candidates = ["label", "target", "class", "y"]

    path_col = next((c for c in path_candidates if c in df.columns), None)
    label_col = next((c for c in label_candidates if c in df.columns), None)

    if path_col is None or label_col is None:
        raise ValueError(f"CSV must include path and label columns. Found: {list(df.columns)}")

    out = df[[path_col, label_col]].copy()
    out.columns = ["file_path", "label"]
    out["label"] = out["label"].astype(int)
    return out

# Build basename index once for robust path recovery
all_wavs = list(data_root.rglob("*.wav"))
basename_to_paths = {}
for p in all_wavs:
    basename_to_paths.setdefault(p.name, []).append(p)

def resolve_audio_path(raw_path: str, data_root: Path) -> Path:
    s = str(raw_path).replace('\\', '/').strip()

    p = Path(s)
    if p.exists():
        return p

    parts = s.split('/')
    if 'data' in parts:
        idx = parts.index('data')
        candidate = data_root / Path(*parts[idx + 1:])
        if candidate.exists():
            return candidate

    for machine_name in ['Machine 1', 'Machine 2', 'Machine 3']:
        marker = f'/{machine_name}/'
        if marker in s:
            suffix = s.split(marker, 1)[1]
            candidate = data_root / machine_name / suffix
            if candidate.exists():
                return candidate

    base = Path(s).name
    matches = basename_to_paths.get(base, [])
    if len(matches) == 1:
        return matches[0]

    return Path(s)

def infer_label_from_path(p: Path) -> int:
    s = str(p).replace('\\', '/').lower()

    if '/machine 1/' in s:
        machine_idx = 0
    elif '/machine 2/' in s:
        machine_idx = 1
    elif '/machine 3/' in s:
        machine_idx = 2
    else:
        raise ValueError(f"Cannot infer machine id from path: {p}")

    if '/abnormal/' in s:
        abnormal = 1
    elif '/normal/' in s:
        abnormal = 0
    else:
        raise ValueError(f"Cannot infer class normal/abnormal from path: {p}")

    return machine_idx * 2 + abnormal

target_len = int(cfg.target_sr * cfg.clip_seconds)
mel_transform = T.MelSpectrogram(
    sample_rate=cfg.target_sr,
    n_fft=cfg.n_fft,
    hop_length=cfg.hop_length,
    n_mels=cfg.n_mels,
    f_min=0,
    f_max=cfg.target_sr // 2,
)
to_db = T.AmplitudeToDB(stype="power")

def fit_length(waveform: torch.Tensor, target_length: int) -> torch.Tensor:
    cur_len = waveform.shape[1]
    if cur_len < target_length:
        waveform = torch.nn.functional.pad(waveform, (0, target_length - cur_len))
    elif cur_len > target_length:
        start = (cur_len - target_length) // 2
        waveform = waveform[:, start:start + target_length]
    return waveform

def preprocess_waveform(waveform: torch.Tensor, sr: int) -> torch.Tensor:
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != cfg.target_sr:
        waveform = torchaudio.functional.resample(waveform, sr, cfg.target_sr)

    if cfg.remove_dc_offset:
        waveform = waveform - waveform.mean(dim=1, keepdim=True)

    if cfg.peak_normalize_waveform:
        peak = waveform.abs().amax(dim=1, keepdim=True).clamp_min(1e-8)
        waveform = waveform / peak

    if cfg.pre_emphasis:
        x0 = waveform[:, :1]
        x1 = waveform[:, 1:] - cfg.pre_emphasis_coef * waveform[:, :-1]
        waveform = torch.cat([x0, x1], dim=1)

    waveform = fit_length(waveform, target_len)
    return waveform

def preprocess_mel(mel: torch.Tensor) -> torch.Tensor:
    mel = torch.clamp(mel, min=cfg.mel_floor_db, max=cfg.mel_ceil_db)

    if cfg.mel_scale_mode == "minmax":
        denom = max(cfg.mel_ceil_db - cfg.mel_floor_db, 1e-8)
        mel = (mel - cfg.mel_floor_db) / denom
    elif cfg.mel_scale_mode == "zscore":
        mu = mel.mean()
        sigma = mel.std().clamp_min(1e-6)
        mel = (mel - mu) / sigma
    else:
        raise ValueError("mel_scale_mode must be one of: minmax, zscore")

    return mel

def waveform_to_mel_numpy(audio_path: Path) -> np.ndarray:
    waveform, sr = torchaudio.load(str(audio_path))
    waveform = preprocess_waveform(waveform, sr)
    mel = to_db(mel_transform(waveform))
    mel = preprocess_mel(mel)
    return mel.squeeze(0).cpu().numpy().astype(np_dtype, copy=False)

dtype_map = {"float16": np.float16, "float32": np.float32}
if cfg.save_dtype not in dtype_map:
    raise ValueError("save_dtype must be one of: float16, float32")
np_dtype = dtype_map[cfg.save_dtype]

## 4) Build Unified Manifest (CSV if available, else folder-derived split)
This avoids hard dependency on CSV files by deriving labels and splits from folder structure when needed.

In [ ]:
csv_paths = {
    'train': Path(cfg.train_csv),
    'val': Path(cfg.val_csv),
    'test': Path(cfg.test_csv),
}

use_csv = cfg.use_csv_splits_if_available and all(p.exists() for p in csv_paths.values())

if use_csv:
    print('Using provided split CSV files.')
    rows = []
    for split_name, csv_path in csv_paths.items():
        df = standardize_columns(pd.read_csv(csv_path))
        df['split'] = split_name
        rows.append(df)

    all_df = pd.concat(rows, axis=0, ignore_index=True)
    all_df['audio_path'] = all_df['file_path'].apply(lambda x: str(resolve_audio_path(x, data_root)))
    all_df = all_df[all_df['audio_path'].apply(lambda p: Path(p).exists())].reset_index(drop=True)
else:
    print('CSV splits not used. Deriving labels from folder names and creating stratified split.')

    wav_paths = sorted(data_root.rglob('*.wav'))
    if len(wav_paths) == 0:
        raise ValueError(f'No WAV files found under {data_root}')

    base_df = pd.DataFrame({'audio_path': [str(p) for p in wav_paths]})
    base_df['label'] = base_df['audio_path'].apply(lambda p: infer_label_from_path(Path(p)))

    # Stratified split by class labels
    rng = np.random.RandomState(cfg.split_seed)
    split_parts = []
    for label, group in base_df.groupby('label'):
        idx = np.arange(len(group))
        rng.shuffle(idx)
        group = group.iloc[idx].reset_index(drop=True)

        n = len(group)
        n_train = int(round(n * cfg.train_ratio))
        n_val = int(round(n * cfg.val_ratio))
        n_train = min(max(n_train, 1), max(n - 2, 1)) if n >= 3 else max(n - 1, 0)
        n_val = min(max(n_val, 1), max(n - n_train - 1, 0)) if n - n_train >= 2 else max(n - n_train, 0)
        n_test = n - n_train - n_val

        train_part = group.iloc[:n_train].copy()
        val_part = group.iloc[n_train:n_train + n_val].copy()
        test_part = group.iloc[n_train + n_val:].copy()

        train_part['split'] = 'train'
        val_part['split'] = 'val'
        test_part['split'] = 'test'

        split_parts.extend([train_part, val_part, test_part])

    all_df = pd.concat(split_parts, axis=0, ignore_index=True)

print(f"Resolved rows: {len(all_df)}")
display(all_df.groupby('split').size())
display(all_df.groupby(['split', 'label']).size().unstack(fill_value=0))

## 5) Convert all WAVs to mel `.npy` (deduplicated)

In [ ]:
out_root = Path(cfg.out_root)
mel_root = out_root / 'mel_npy'
mel_root.mkdir(parents=True, exist_ok=True)

unique_audio = sorted(set(all_df['audio_path'].tolist()))
print(f"Unique audio files to process: {len(unique_audio)}")

audio_to_mel = {}

for audio_str in tqdm(unique_audio, desc='Precompute mel'):
    audio_path = Path(audio_str)

    # Keep relative directory structure under mel_npy for traceability
    rel = audio_path.relative_to(data_root) if audio_path.is_relative_to(data_root) else Path(audio_path.name)
    mel_path = mel_root / rel.with_suffix('.npy')
    mel_path.parent.mkdir(parents=True, exist_ok=True)

    if not mel_path.exists():
        mel_np = waveform_to_mel_numpy(audio_path)
        np.save(mel_path, mel_np)

    audio_to_mel[audio_str] = str(mel_path)

print('Mel precompute done.')

## 6) Export split/class folders with `.npy` files
This creates a folderized dataset layout: `train/<class>/`, `val/<class>/`, `test/<class>/`.

In [ ]:
import hashlib
import shutil

all_df['mel_npy_path'] = all_df['audio_path'].map(audio_to_mel)

# Label mapping to requested folder naming convention.
label_to_folder = {
    0: 'machine1-normal',
    1: 'machine1-abnormal',
    2: 'machine2-normal',
    3: 'machine2-abnormal',
    4: 'machine3-normal',
    5: 'machine3-abnormal',
}

split_root = out_root / 'splits'
split_root.mkdir(parents=True, exist_ok=True)

rows_out = []
for row in all_df.itertuples(index=False):
    split_name = str(row.split)
    label = int(row.label)
    src_npy = Path(row.mel_npy_path)

    class_folder = label_to_folder.get(label, f'class-{label}')
    dst_dir = split_root / split_name / class_folder
    dst_dir.mkdir(parents=True, exist_ok=True)

    # Stable unique suffix avoids filename collisions across similarly named source files.
    uid = hashlib.md5(str(src_npy).encode('utf-8')).hexdigest()[:10]
    dst_npy = dst_dir / f"{src_npy.stem}_{uid}.npy"

    if not dst_npy.exists():
        shutil.copy2(src_npy, dst_npy)

    rows_out.append({
        'split': split_name,
        'label': label,
        'class_folder': class_folder,
        'feature_path': str(dst_npy),
    })

folder_manifest_df = pd.DataFrame(rows_out)

# Optional index CSVs for auditing only (training can ignore these).
manifest_root = out_root / 'manifests'
manifest_root.mkdir(parents=True, exist_ok=True)
for split_name in ['train', 'val', 'test']:
    split_df = folder_manifest_df[folder_manifest_df['split'] == split_name].copy()
    out_csv = manifest_root / f'{split_name}_folder_index.csv'
    split_df.to_csv(out_csv, index=False)
    print(f"Wrote index: {out_csv} ({len(split_df)} rows)")

cfg_out = out_root / 'precompute_config.json'
with open(cfg_out, 'w', encoding='utf-8') as f:
    json.dump(asdict(cfg), f, indent=2)

readme_path = out_root / 'README.txt'
readme_text = (
    'This dataset contains precomputed mel spectrogram .npy files in split/class folders.\n'
    'Layout: splits/train/<class>/*.npy, splits/val/<class>/*.npy, splits/test/<class>/*.npy\n'
    'Class folders: machine1-normal, machine1-abnormal, machine2-normal, machine2-abnormal, machine3-normal, machine3-abnormal\n'
    'Optional index CSVs are provided under manifests/.\n'
)
readme_path.write_text(readme_text, encoding='utf-8')

print(f"Output root: {out_root}")
print(f"Split root:  {split_root}")
print('Counts by split/class:')
display(folder_manifest_df.groupby(['split', 'class_folder']).size().unstack(fill_value=0))

## 7) Kaggle API authentication (for publish)

Set Kaggle secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` in Notebook secrets before running this cell.

In [ ]:
if cfg.publish:
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        os.environ['KAGGLE_USERNAME'] = secrets.get_secret('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = secrets.get_secret('KAGGLE_KEY')

        if not os.environ.get('KAGGLE_USERNAME') or not os.environ.get('KAGGLE_KEY'):
            raise ValueError('KAGGLE_USERNAME/KAGGLE_KEY secrets are empty.')

        print('Kaggle API credentials loaded from secrets.')
    except Exception as e:
        raise RuntimeError(
            'Failed to load Kaggle API credentials from secrets. ' 
            'Set KAGGLE_USERNAME and KAGGLE_KEY in Kaggle secrets.'
        ) from e
else:
    print('Publishing is disabled by config (cfg.publish=False).')

## 8) Publish dataset to Kaggle (create or version)

In [ ]:
if cfg.publish:
    dataset_ref = f"{cfg.kaggle_owner}/{cfg.kaggle_dataset_slug}"

    metadata = {
        'title': cfg.kaggle_title,
        'id': dataset_ref,
        'subtitle': cfg.kaggle_subtitle,
        'description': cfg.kaggle_description,
        'licenses': [{'name': cfg.kaggle_license}]
    }

    metadata_path = Path(cfg.out_root) / 'dataset-metadata.json'
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

    # Try create first; if already exists, create a new version
    create_cmd = [
        'kaggle', 'datasets', 'create',
        '-p', str(cfg.out_root),
        '-u',
    ]

    version_cmd = [
        'kaggle', 'datasets', 'version',
        '-p', str(cfg.out_root),
        '-m', 'Update precomputed mel npy dataset',
        '-u',
    ]

    print('Attempting: kaggle datasets create ...')
    res = subprocess.run(create_cmd, capture_output=True, text=True)

    if res.returncode == 0:
        print('Dataset created successfully.')
        print(res.stdout)
    else:
        print('Create failed, trying version update...')
        print(res.stderr)
        res2 = subprocess.run(version_cmd, capture_output=True, text=True)
        if res2.returncode != 0:
            print(res2.stdout)
            print(res2.stderr)
            raise RuntimeError('Kaggle publish failed for both create and version commands.')
        print('Dataset version uploaded successfully.')
        print(res2.stdout)

    print(f"Published dataset ref: {dataset_ref}")
    print(f"Expected URL: https://www.kaggle.com/datasets/{dataset_ref}")
else:
    print('Skipping publish step because cfg.publish=False')

## 9) Notes

- Precomputing mel `.npy` usually reduces CPU preprocessing overhead and improves GPU utilization during training.
- Output layout is folderized for direct loading without split CSV dependency:
  - `splits/train/<class>/*.npy`
  - `splits/val/<class>/*.npy`
  - `splits/test/<class>/*.npy`
- Optional index CSVs are still written under `manifests/` for audit/debug.
- If your Kaggle notebook has Internet disabled, publish step will fail until Internet is enabled.